# 8. K-Nearest Neighbors (mockup)

**KNN regression** forecasts a quarter by finding the **K training quarters with the most similar lagged features** and averaging their charge-off rates.
- **Scaling is required:** distance is computed across all features, so unscaled features with large units would dominate (StandardScaler, fit on training folds only).
- **K** controls the bias-variance trade-off: a small K follows individual quarters (overfits), a large K smooths toward the average (underfits).
- **weights:** `uniform` gives every neighbor an equal vote; `distance` gives closer neighbors more weight.
- Like trees, KNN **cannot extrapolate**: it can only average charge-off rates seen in training, which matters for 2020–21.
- **Note:** with `weights="distance"`, each training quarter is its own nearest neighbor (distance 0), so train RMSE is always 0. Judge overfitting from the CV and test RMSE, not the train-test gap.

In [ ]:
from prep import *
from sklearn.neighbors import KNeighborsRegressor

X_train, X_test, y_train, y_test, meta = load_model_data()

knn_pipe = make_pipeline(*preprocess(scale=True), KNeighborsRegressor())
knn_grid = {"kneighborsregressor__n_neighbors": list(range(1, 16)),
            "kneighborsregressor__weights": ["uniform", "distance"]}
knn_search = tune(knn_pipe, knn_grid, X_train, y_train)   # TimeSeriesSplit(5), RMSE
knn_best = knn_search.best_estimator_
plot_tuning(knn_search, "kneighborsregressor__n_neighbors", by="kneighborsregressor__weights", logx=False);

In [ ]:
display(score(knn_best, X_train, X_test, y_train, y_test))
plot_forecast(knn_best, X_train, X_test, y_train, y_test, title="KNN: actual vs forecast")
plot_permutation(permutation_table(knn_best, X_train, y_train, X_test, y_test), top=15)   # KNN has no weights
record("KNN", knn_best, X_train, X_test, y_train, y_test, search=knn_search, notes=str(knn_search.best_params_))